# HW 4: Dynamic Programming

Code up the dynamic programming problem for consumers who are forward-looking about getting "locked-in" to one of the yogurt brands.

To keep things simple, ignore feature and price, assuming the only explanatory variable is the brand they are locked-in to.

Keep in mind that the actual data set should play no role in your coding, other than understanding the nature of the problem.  You are simply calculating the future/continuation values associated with each choice.

In [1]:
import pandas as pd
import numpy as np
from numpy import euler_gamma
from scipy.special import logsumexp

In [2]:
# Parameters (from homogeneous MNL; see notebook 2)

beta_0 = np.array([0, -0.699019, -1.392080, -3.825416, -1.254406])
beta_b = np.array([2.466313])

delta = 0.99

In [3]:
states = np.array([0, 1, 2, 3, 4])    # what brand is the customer "locked-in" to?
actions = np.array([0, 1, 2, 3, 4])   # what brand does the customer choose to buy?

In [4]:
# Current period payoffs: rows are actions, columns are states

Imat = np.eye(5, dtype=int)
Imat[0, 0] = 0
U = beta_0[:, None] + beta_b * Imat

Imat, U

(array([[0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1]]),
 array([[ 0.      ,  0.      ,  0.      ,  0.      ,  0.      ],
        [-0.699019,  1.767294, -0.699019, -0.699019, -0.699019],
        [-1.39208 , -1.39208 ,  1.074233, -1.39208 , -1.39208 ],
        [-3.825416, -3.825416, -3.825416, -1.359103, -3.825416],
        [-1.254406, -1.254406, -1.254406, -1.254406,  1.211907]]))

In [5]:
# Transition matrices: (s, a, s') = (current state, action, next state)

Tmats = np.zeros((5, 5, 5))
Tmats[:, 1:, 1:] = np.eye(4)
Tmats[np.arange(5), 0, np.arange(5)] = 1

Tmats

array([[[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]],

       [[0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]],

       [[0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]],

       [[0., 0., 0., 1., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]],

       [[0., 0., 0., 0., 1.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]]])

In [6]:
# For illustration: what states do we transition to if we take each action in each state?

Z = np.tile(np.arange(5), (5, 1)).T
Z1 = (Tmats @ Z)[:, :, 0].T
Z1

array([[0., 1., 2., 3., 4.],
       [1., 1., 1., 1., 1.],
       [2., 2., 2., 2., 2.],
       [3., 3., 3., 3., 3.],
       [4., 4., 4., 4., 4.]])

In [7]:
def Q_function(Nu):
    """
    Calculates the Q function given the current guess of Nu.

    u(a,s) = beta_0^T 1{a} + beta_b 1{a=s}
    U_ij = u(i,j)

    nu(a,s) = u(a,s) + delta * (gamma + log-sum-exp_{a'}(nu(a',a)))
    Nu_ij = nu(i,j)
    
    Shape: (n_a, n_s) = (5, 5)
    """
    Imat = np.eye(5, dtype=int)
    Imat[0, 0] = 0
    U = beta_0[:, None] + beta_b * Imat
    
    cont_vals = logsumexp(Nu, axis=0)
    Nu_curr = U + delta * (euler_gamma + (Tmats @ cont_vals).T)
    return Nu_curr

def find_fixed_point(Nu_init, tol=1e-12):
    Nu_curr = Nu_init
    diff = np.inf
    while diff > tol:
        Nu_next = Q_function(Nu_curr)
        diff = np.linalg.norm(Nu_next - Nu_curr, ord='fro')
        Nu_curr = Nu_next
    return Nu_curr

In [8]:
Nu_init = np.full((5, 5), 0)
Nu = find_fixed_point(Nu_init)
Nu = pd.DataFrame(Nu)
print(Nu)

            0           1           2           3           4
0  247.074398  249.329032  247.679350  247.115193  247.809218
1  248.630013  251.096326  248.630013  248.630013  248.630013
2  246.287270  246.287270  248.753583  246.287270  246.287270
3  243.289777  243.289777  243.289777  245.756090  243.289777
4  246.554812  246.554812  246.554812  246.554812  249.021125


In [9]:
print((Nu - U)/delta)

            0           1           2           3           4
0  249.570099  251.847507  250.181162  249.611306  250.312341
1  251.847507  251.847507  251.847507  251.847507  251.847507
2  250.181162  250.181162  250.181162  250.181162  250.181162
3  249.611306  249.611306  249.611306  249.611306  249.611306
4  250.312341  250.312341  250.312341  250.312341  250.312341
